In [ ]:
import importlib
import bankhuman.database
import bankhuman.models
import plotly.express as px

bankhuman.models = importlib.reload(bankhuman.models)
bankhuman.database = importlib.reload(bankhuman.database)
from bankhuman.models import Product, ProductType, Registry
import pandas as pd
#parser.add_argument("--db", default="data/bankhuman.db", help="SQLite database path")
#args = parser.parse_args()
db = bankhuman.database.Database("data/bankhuman.db")


In [11]:
pd.DataFrame(db.list_products())

,name,product_type,currency,institution,labels,metadata,iban,isin,owner,id
0,Amazon,ProductType.INVESTMENT,EUR,Revolut,(),{},NaN,NaN,yo,20
1,AMD,ProductType.INVESTMENT,EUR,Revolut,(),{},NaN,NaN,yo,22
2,bono americano,ProductType.OTHER,EUR,Revolut,(),{},NaN,NaN,yo,23
3,BYD,ProductType.INVESTMENT,EUR,Trade Republic,(),"{'import_section': 'investment', 'source_alias...",NaN,NaN,yo,14
4,Casa,ProductType.OTHER,EUR,NaN,(),"{'import_section': 'property', 'source_alias':...",NaN,NaN,yo,18
5,"Depósito 2,5% 4 meses",ProductType.OTHER,EUR,ING,(),"{'import_section': 'deposit', 'source_alias': ...",9808496610,NaN,yo,13
6,Fondo Euro Stoxx 50,ProductType.FUND,EUR,ING,(),"{'import_section': 'fund', 'source_alias': 'Fo...",NaN,ES0152771038,yo,10
7,Fondo Ibex 35,ProductType.FUND,EUR,ING,(),"{'import_section': 'fund', 'source_alias': 'Fo...",NaN,ES0152741031,yo,11
8,Fondo S&P 500,ProductType.FUND,EUR,ING,(),"{'import_section': 'fund', 'source_alias': 'Fo...",NaN,ES0152769032,yo,12
9,Health,ProductType.BANK_ACCOUNT,EUR,B100,(),{'interest_rate': 2.5},NaN,NaN,yo,3


In [12]:
print(f" Atributos: {list(db.__dict__.keys())}")

metodos = [
nombre for nombre in dir(db)
if callable(getattr(db, nombre))
and not nombre.startswith('__')
]
print(f" Métodos: {metodos}")

 Atributos: ['path']
 Métodos: ['_import_position_global', 'add_product', 'add_registry', 'connect', 'delete_product', 'delete_registry', 'find_product_by_identifier', 'import_excel', 'initialize', 'latest_registries', 'list_products', 'list_registries', 'update_product']


In [13]:
registered_product_ids = {
    registry.product_id
    for registry, _ in db.list_registries(limit=100000)
}

products_without_registers = [
    product
    for product in db.list_products()
    if product.id not in registered_product_ids
]

pd.DataFrame(products_without_registers)

""


In [ ]:
latest_records = db.latest_registries()
historical_records = db.list_registries(limit=100000)

def records_dataframe(records):
    return pd.DataFrame([
        {
            "registry_id": registry.id,
            "product_id": product.id,
            "date": registry.recorded_on,
            "amount": float(registry.amount),
            "currency": product.currency,
            "product": product.name,
            "product_type": product.product_type.value,
            "owner": product.owner,
            "institution": product.institution,
        }
        for registry, product in records
    ])

latest_df_all = records_dataframe(latest_records)
historical_df_all = records_dataframe(historical_records)
latest_df_yo = latest_df_all[latest_df_all["owner"].str.casefold() == "yo"].copy()
historical_df_yo = historical_df_all[historical_df_all["owner"].str.casefold() == "yo"].copy()

latest_df = latest_df_yo[
    (latest_df_yo["product_type"] != "other")
    & (latest_df_yo["product"].str.casefold() != "casa")
].copy()
historical_df = historical_df_yo[
    (historical_df_yo["product_type"] != "other")
    & (historical_df_yo["product"].str.casefold() != "casa")
].copy()
latest_df


,registry_id,product_id,date,amount,currency,product,product_type,owner,institution
0,21,3,2026-09-15,2481.87,EUR,Health,bank_account,yo,B100
1,4,8,2026-09-14,103.46,EUR,ING (5075),bank_account,yo,ING
3,11,15,2026-09-14,253.29,EUR,Personal,bank_account,yo,Bankinter
4,22,5,2026-09-15,7.65,EUR,Save,bank_account,yo,B100
5,3,7,2026-09-14,7802.65,EUR,Trade Republic (1611),bank_account,yo,Trade Republic
6,12,16,2026-09-14,-15.89,EUR,Visa Clasica,credit_card,yo,NaN
7,6,10,2026-09-14,1436.07,EUR,Fondo Euro Stoxx 50,fund,yo,ING
8,7,11,2026-09-14,1471.12,EUR,Fondo Ibex 35,fund,yo,ING
9,8,12,2026-09-14,2123.40,EUR,Fondo S&P 500,fund,yo,ING
10,15,20,2026-09-15,917.23,EUR,Amazon,investment,yo,Revolut


## Total amount

In [16]:
type_by_currency = (
    latest_df.groupby(["currency", "product_type"], as_index=False)["amount"]
    .sum()
)

fig = px.bar(
    type_by_currency,
    x="currency",
    y="amount",
    color="product_type",
    title="Current total amount by currency and product type",
    labels={"currency": "Currency", "amount": "Amount", "product_type": "Product type"},
    hover_data={"amount": ":,.2f"},
)
fig.update_layout(barmode="stack", legend_title_text="Product type")
fig.show()


## Current amount by type

In [17]:
type_by_institution = (
    latest_df[latest_df["product_type"] != "other"]
    .assign(institution=lambda dataframe: dataframe["institution"].fillna("Unknown"))
    .groupby(["product_type", "currency", "institution"], as_index=False)["amount"]
    .sum()
)
type_by_institution["label"] = (
    type_by_institution["product_type"]
    + " ("
    + type_by_institution["currency"]
    + ")"
)

fig = px.bar(
    type_by_institution,
    x="label",
    y="amount",
    color="institution",
    barmode="stack",
    title="Current amount by product type and institution",
    labels={
        "label": "Product type",
        "amount": "Amount",
        "institution": "Institution",
    },
    hover_data={
        "amount": ":,.2f",
        "product_type": True,
        "currency": True,
    },
)
fig.update_xaxes(tickangle=45)
fig.show()


## Current amount by institution

In [18]:
institution_by_type = (
    latest_df[latest_df["product_type"] != "other"]
    .assign(institution=lambda dataframe: dataframe["institution"].fillna("Unknown"))
    .groupby(["institution", "currency", "product_type"], as_index=False)["amount"]
    .sum()
)
institution_by_type["label"] = (
    institution_by_type["institution"]
    + " ("
    + institution_by_type["currency"]
    + ")"
)

fig = px.bar(
    institution_by_type,
    x="label",
    y="amount",
    color="product_type",
    barmode="stack",
    title="Current amount by institution and product type",
    labels={
        "label": "Institution",
        "amount": "Amount",
        "product_type": "Product type",
    },
    hover_data={"amount": ":,.2f", "currency": True},
)
fig.update_xaxes(tickangle=45)
fig.show()


## Evolution by product

In [23]:
evolution = (
    historical_df.groupby(["date", "product", "currency"], as_index=False)["amount"]
    .sum()
    .sort_values("date")
)
evolution["label"] = evolution["product"] + " (" + evolution["currency"] + ")"
evolution["series"] = "Product"

product_snapshot_dates = sorted(historical_df["date"].unique())
product_snapshots = []
for snapshot_date in product_snapshot_dates:
    latest_at_date = (
        historical_df[historical_df["date"] <= snapshot_date]
        .sort_values(["date", "registry_id"])
        .drop_duplicates("product_id", keep="last")
        .copy()
    )
    latest_at_date["date"] = snapshot_date
    product_snapshots.append(latest_at_date)

latest_products_by_date = pd.concat(product_snapshots, ignore_index=True)
product_totals = (
    latest_products_by_date.groupby(["date", "currency"], as_index=False)["amount"]
    .sum()
    .assign(product="Total", product_type="Total")
)
product_totals["label"] = "Total (" + product_totals["currency"] + ")"
product_totals["series"] = "Total"
evolution_with_totals = pd.concat([evolution, product_totals], ignore_index=True)

fig = px.line(
    evolution_with_totals,
    x="date",
    y="amount",
    color="label",
    line_dash="series",
    markers=True,
    title="Temporal evolution by product",
    labels={"date": "Date", "amount": "Amount", "label": "Product"},
    hover_data={"amount": ":,.2f", "product": True, "currency": True},
)
fig.update_traces(selector={"name": "Total (EUR)"}, line={"width": 4})
fig.show()


## Evolution by type

In [22]:
snapshot_dates = sorted(historical_df["date"].unique())
snapshots = []

for snapshot_date in snapshot_dates:
    latest_at_date = (
        historical_df[historical_df["date"] <= snapshot_date]
        .sort_values(["date", "registry_id"])
        .drop_duplicates("product_id", keep="last")
        .copy()
    )
    latest_at_date["date"] = snapshot_date
    snapshots.append(latest_at_date)

latest_by_date = pd.concat(snapshots, ignore_index=True)
type_evolution = (
    latest_by_date.groupby(["date", "product_type", "currency"], as_index=False)["amount"]
    .sum()
    .sort_values("date")
)
type_evolution["label"] = (
    type_evolution["product_type"] + " (" + type_evolution["currency"] + ")"
)
type_evolution["series"] = "Product type"

type_totals = (
    type_evolution.groupby(["date", "currency"], as_index=False)["amount"]
    .sum()
    .assign(product_type="Total")
)
type_totals["label"] = "Total (" + type_totals["currency"] + ")"
type_totals["series"] = "Total"
type_evolution_with_totals = pd.concat([type_evolution, type_totals], ignore_index=True)

fig = px.line(
    type_evolution_with_totals,
    x="date",
    y="amount",
    color="label",
    line_dash="series",
    markers=True,
    title="Temporal evolution by product type",
    labels={"date": "Date", "amount": "Amount", "label": "Product type"},
    hover_data={"amount": ":,.2f", "product_type": True, "currency": True},
)
fig.update_traces(selector={"name": "Total (EUR)"}, line={"width": 4})
fig.show()
